# Student Helpdesk Email Classification
## CT-264 Programming for Artificial Intelligence — CCA

**Problem:** Automatically triage student support emails by predicting their **category** and **priority**.

**Datasets used:**
- **Dataset 1 (Real):** 20 labelled student emails from `data/inbox/` (ground-truth labels from `outbox.json`).
- **Dataset 2 (Synthetic):** 400 programmatically generated emails covering all 9 categories.

**Approach:**
1. Exploratory Data Analysis (EDA) — distribution, class imbalance, text length, word frequency.
2. Text preprocessing — cleaning, TF-IDF vectorisation.
3. Model training — Logistic Regression, Naive Bayes, Linear SVM.
4. Evaluation — accuracy, F1-score, confusion matrix, cross-validation.
5. Comparison with the full LLM-based triage agent.

---

## 1. Setup

In [ ]:
# Install required packages (run once)
%pip install -q pandas numpy matplotlib seaborn scikit-learn
print('All packages ready.')

In [ ]:
import json
import string
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='husl')
print('Libraries loaded.')

In [ ]:
# Resolve paths regardless of where Jupyter is launched from
def _find_root() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'data' / 'inbox').exists():
            return candidate
    raise FileNotFoundError('Cannot find project root (data/inbox/ not found). Launch Jupyter from the repo root.')

ROOT        = _find_root()
INBOX_DIR   = ROOT / 'data' / 'inbox'
OUTBOX_PATH = ROOT / 'outbox.json'
SYNTHETIC   = ROOT / 'analysis' / 'data' / 'synthetic_emails.csv'
PLOTS_DIR   = ROOT / 'analysis' / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {ROOT}')
print(f'Inbox        : {INBOX_DIR}')
print(f'Outbox       : {OUTBOX_PATH}')
print(f'Synthetic    : {SYNTHETIC}')
print(f'Plots dir    : {PLOTS_DIR}')

---
## 2. Load Datasets

### 2.1 Dataset 1 — Real Emails (from project inbox)

In [ ]:
outbox_records = json.loads(OUTBOX_PATH.read_text(encoding='utf-8'))
outbox_map = {r['email_id']: r for r in outbox_records}

real_rows = []
for path in sorted(INBOX_DIR.glob('E*.json')):
    email = json.loads(path.read_text(encoding='utf-8'))
    rec = outbox_map.get(email['id'])
    if rec:
        real_rows.append({
            'subject':  email.get('subject', ''),
            'body':     email.get('body', ''),
            'category': rec['category'],
            'priority': rec['priority'],
            'escalate': rec['escalate'],
            'source':   'real',
        })

df_real = pd.DataFrame(real_rows)
print(f'Dataset 1 — Real emails: {len(df_real)} rows')
display(df_real.head(3))

### 2.2 Dataset 2 — Synthetic Emails

Generated by `analysis/generate_data.py` covering all 9 categories.

In [ ]:
if not SYNTHETIC.exists():
    print('Synthetic CSV not found. Generating now...')
    import subprocess, sys
    subprocess.run([sys.executable, str(ROOT / 'analysis' / 'generate_data.py')], check=True)

df_syn = pd.read_csv(SYNTHETIC)
print(f'Dataset 2 — Synthetic emails: {len(df_syn)} rows')
display(df_syn.head(3))

### 2.3 Combine Datasets

In [ ]:
df = pd.concat([df_real, df_syn], ignore_index=True)
df['text']        = (df['subject'].fillna('') + ' ' + df['body'].fillna('')).str.strip()
df['text_length'] = df['text'].str.len()
df['word_count']  = df['text'].str.split().str.len()

print(f'Combined dataset : {len(df)} rows')
print(f'Categories       : {sorted(df["category"].unique())}')
print(f'Missing values   :\n{df[["subject","body","category","priority"]].isnull().sum().to_string()}')

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

In [ ]:
print('=' * 55)
print('DATASET OVERVIEW')
print('=' * 55)
print(f'Total samples         : {len(df)}')
print(f'Real emails           : {(df["source"] == "real").sum()}')
print(f'Synthetic emails      : {(df["source"] == "synthetic").sum()}')
print(f'Unique categories     : {df["category"].nunique()}')
print(f'Escalation rate       : {df["escalate"].mean():.1%}')
print()
print('Category breakdown:')
print(df['category'].value_counts().to_string())
print()
print('Priority breakdown:')
print(df['priority'].value_counts().to_string())
print()
print('Word count statistics:')
print(df['word_count'].describe().round(1).to_string())

### 3.2 Category Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall distribution
cat_counts = df['category'].value_counts()
colors = sns.color_palette('husl', len(cat_counts))
bars = axes[0].bar(cat_counts.index, cat_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Category Distribution — Combined Dataset', fontweight='bold', pad=12)
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Number of Emails')
axes[0].tick_params(axis='x', rotation=45)
for bar in bars:
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        str(int(bar.get_height())),
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

# Split by source
source_cat = df.groupby(['source', 'category']).size().unstack(fill_value=0)
source_cat.T.plot(kind='bar', ax=axes[1], color=['#2196F3', '#FF9800'], edgecolor='white', width=0.75)
axes[1].set_title('Category Distribution by Source', fontweight='bold', pad=12)
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Source')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'category_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: category_distribution.png')

### 3.3 Priority Distribution

In [ ]:
PRIORITY_ORDER   = ['low', 'medium', 'high']
PRIORITY_PALETTE = {'low': '#4CAF50', 'medium': '#FF9800', 'high': '#F44336'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall
pri_counts = df['priority'].value_counts().reindex(PRIORITY_ORDER)
axes[0].bar(pri_counts.index, pri_counts.values,
            color=[PRIORITY_PALETTE[p] for p in pri_counts.index], edgecolor='white')
axes[0].set_title('Priority Distribution', fontweight='bold', pad=12)
axes[0].set_xlabel('Priority')
axes[0].set_ylabel('Count')
for i, v in enumerate(pri_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Priority by category
pivot = df.groupby(['category', 'priority']).size().unstack(fill_value=0)
for col in PRIORITY_ORDER:
    if col not in pivot.columns:
        pivot[col] = 0
pivot[PRIORITY_ORDER].plot(
    kind='bar', ax=axes[1],
    color=[PRIORITY_PALETTE[p] for p in PRIORITY_ORDER], edgecolor='white', width=0.75
)
axes[1].set_title('Priority Breakdown by Category', fontweight='bold', pad=12)
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Priority')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'priority_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: priority_distribution.png')

### 3.4 Class Imbalance Analysis

Class imbalance occurs when some categories have far more samples than others, which can bias the classifier.

In [ ]:
cat_counts   = df['category'].value_counts()
majority_cls = cat_counts.idxmax()
minority_cls = cat_counts.idxmin()
IR = cat_counts.max() / cat_counts.min()

print('Class Imbalance Report')
print('=' * 50)
print(f'Majority class  : {majority_cls} ({cat_counts.max()} samples)')
print(f'Minority class  : {minority_cls} ({cat_counts.min()} samples)')
print(f'Imbalance Ratio : {IR:.2f} : 1')
print()
if IR > 10:
    print('HIGH imbalance — will use class_weight="balanced" in classifiers.')
elif IR > 3:
    print('MODERATE imbalance — class_weight="balanced" applied as a safeguard.')
else:
    print('LOW imbalance — dataset is reasonably balanced.')

print()
print(f'  {"Category":<15} {"Count":>7}  {"Share":>8}  Bar')
print('  ' + '-' * 48)
for cat, count in cat_counts.items():
    pct  = count / len(df) * 100
    bar  = chr(9608) * int(pct / 1.5)
    print(f'  {cat:<15} {count:>7}  {pct:>7.1f}%  {bar}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E53935' if c == minority_cls else ('#43A047' if c == majority_cls else '#5C6BC0')
          for c in cat_counts.index]
bars = ax.bar(cat_counts.index, cat_counts.values, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(cat_counts.mean(), linestyle='--', color='grey', linewidth=1.2, label=f'Mean ({cat_counts.mean():.0f})')
ax.set_title('Class Imbalance — Sample Count per Category', fontweight='bold', pad=12)
ax.set_xlabel('Category')
ax.set_ylabel('Sample Count')
ax.tick_params(axis='x', rotation=45)
ax.legend()
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'class_imbalance.png', bbox_inches='tight')
plt.show()
print('Saved: class_imbalance.png')

### 3.5 Text Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word count histogram
axes[0].hist(df['word_count'], bins=30, color='#5C6BC0', edgecolor='white', alpha=0.85)
median_wc = df['word_count'].median()
axes[0].axvline(median_wc, color='#E53935', linestyle='--', linewidth=1.5,
                label=f'Median: {median_wc:.0f} words')
axes[0].set_title('Email Word Count Distribution', fontweight='bold', pad=12)
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Word count by category — violin plot
cat_order = df.groupby('category')['word_count'].median().sort_values().index.tolist()
df_sorted = df.copy()
df_sorted['category'] = pd.Categorical(df_sorted['category'], categories=cat_order, ordered=True)
df_sorted_grp = df_sorted.sort_values('category')
sns.boxplot(data=df_sorted_grp, x='category', y='word_count', ax=axes[1],
            palette='husl', order=cat_order)
axes[1].set_title('Word Count Distribution by Category', fontweight='bold', pad=12)
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Word Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'text_length.png', bbox_inches='tight')
plt.show()
print('Saved: text_length.png')
print(f'\nWord count summary:\n{df["word_count"].describe().round(1).to_string()}')

### 3.6 Top Words per Category

In [ ]:
# Simple tokeniser — no NLTK dependency needed
STOPWORDS = {
    'i', 'me', 'my', 'we', 'our', 'you', 'your', 'he', 'she', 'it', 'they',
    'this', 'that', 'is', 'am', 'are', 'was', 'were', 'be', 'been', 'have',
    'has', 'had', 'do', 'does', 'did', 'will', 'would', 'can', 'could', 'shall',
    'should', 'may', 'might', 'a', 'an', 'the', 'and', 'but', 'or', 'if', 'in',
    'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'as', 'not', 'no',
    'so', 'just', 'dear', 'sir', 'madam', 'hello', 'hi', 'regards', 'please',
    'thank', 'thanks', 'student', 'helpdesk', 'roll', 'also', 'very', 'more',
    'been', 'than', 'then', 'yet', 'about', 'up', 'out', 'some', 'what', 'which',
    'who', 'how', 'when', 'where', 'there', 'here', 'all', 'any', 'both', 'each',
}
_PUNCT = str.maketrans('', '', string.punctuation)

def top_words(texts: pd.Series, n: int = 10) -> list:
    tokens = []
    for text in texts:
        for word in str(text).lower().translate(_PUNCT).split():
            if word not in STOPWORDS and len(word) > 2 and not word.isdigit():
                tokens.append(word)
    return Counter(tokens).most_common(n)

categories = sorted(df['category'].unique())
palette    = sns.color_palette('husl', len(categories))

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes_flat = axes.flatten()

for idx, cat in enumerate(categories):
    ax    = axes_flat[idx]
    items = top_words(df[df['category'] == cat]['text'])
    if not items:
        ax.set_visible(False)
        continue
    words, counts = zip(*items)
    ax.barh(list(reversed(words)), list(reversed(counts)),
            color=palette[idx], alpha=0.85, edgecolor='white')
    ax.set_title(cat.upper().replace('_', ' '), fontweight='bold', fontsize=10)
    ax.set_xlabel('Frequency')
    ax.tick_params(axis='y', labelsize=8)

for ax in axes_flat[len(categories):]:
    ax.set_visible(False)

plt.suptitle('Top 10 Words per Category (after stopword removal)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'word_frequency.png', bbox_inches='tight')
plt.show()
print('Saved: word_frequency.png')

---
## 4. Data Preprocessing

**Steps:**
1. Combine subject + body into a single `text` field.
2. Lowercase.
3. Remove punctuation.
4. Remove tokens shorter than 3 characters and pure numbers.

No external NLP libraries are required.

In [ ]:
_PUNCT_TABLE = str.maketrans('', '', string.punctuation)

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = text.translate(_PUNCT_TABLE)       # remove punctuation
    tokens = [
        w for w in text.split()
        if len(w) > 2 and not w.isdigit()     # drop short tokens and pure numbers
    ]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(clean_text)

print('Sample preprocessing output:')
for _, row in df.sample(3, random_state=7).iterrows():
    print(f'\n  Category : {row["category"]}')
    print(f'  Original : {row["text"][:100]}...')
    print(f'  Cleaned  : {row["clean_text"][:100]}...')

---
## 5. Model Training

Three classical text classifiers are compared:

| Model | Description |
|---|---|
| **Logistic Regression** | Linear model, effective baseline for text; supports `class_weight` |
| **Multinomial Naive Bayes** | Probabilistic model well-suited to word-count features |
| **Linear SVM** | Strong linear classifier; often best on high-dimensional TF-IDF |

All use a **TF-IDF** (Term Frequency–Inverse Document Frequency) vectoriser with unigrams + bigrams.

In [ ]:
X = df['clean_text']
y_cat = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.20, random_state=42, stratify=y_cat
)

print(f'Train : {len(X_train)} samples')
print(f'Test  : {len(X_test)} samples')
print(f'\nTest set category distribution:')
print(y_test.value_counts().to_string())

In [ ]:
MODELS = {
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)),
        ('clf',   LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
    ]),
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=False)),
        ('clf',   MultinomialNB(alpha=0.1)),
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)),
        ('clf',   LinearSVC(max_iter=2000, class_weight='balanced', random_state=42)),
    ]),
}

results = {}
for name, pipeline in MODELS.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    results[name] = {
        'pipeline':    pipeline,
        'y_pred':      y_pred,
        'accuracy':    accuracy_score(y_test, y_pred),
        'f1_macro':    f1_score(y_test, y_pred, average='macro'),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted'),
    }
    print(f'{name:22s}  Accuracy: {results[name]["accuracy"]:.3f}  '
          f'F1-macro: {results[name]["f1_macro"]:.3f}  '
          f'F1-weighted: {results[name]["f1_weighted"]:.3f}')

In [ ]:
model_names = list(results.keys())
accuracies  = [results[m]['accuracy']    for m in model_names]
f1_macros   = [results[m]['f1_macro']    for m in model_names]
f1_weighted = [results[m]['f1_weighted'] for m in model_names]
bar_colors  = ['#5C6BC0', '#26A69A', '#EF5350']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric_vals, title in zip(
    axes,
    [accuracies, f1_macros, f1_weighted],
    ['Accuracy', 'F1-Score (Macro)', 'F1-Score (Weighted)']
):
    bars = ax.bar(model_names, metric_vals, color=bar_colors, edgecolor='white')
    ax.set_title(title, fontweight='bold', pad=12)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel(title)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, metric_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle('Model Comparison on Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'model_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

best_name = max(results, key=lambda m: results[m]['f1_macro'])
print(f'\nBest model: {best_name}  (F1-macro = {results[best_name]["f1_macro"]:.3f})')

---
## 6. Best Model — Detailed Evaluation

### 6.1 Classification Report

In [ ]:
best_pred = results[best_name]['y_pred']

print(f'Classification Report — {best_name}')
print('=' * 70)
print(classification_report(y_test, best_pred, digits=3))

### 6.2 Confusion Matrix

Shows which categories the model confuses with one another.

In [ ]:
LABELS = sorted(df['category'].unique())
cm     = confusion_matrix(y_test, best_pred, labels=LABELS)

fig, ax = plt.subplots(figsize=(11, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
disp.plot(ax=ax, cmap='Blues', colorbar=True, xticks_rotation=45)
ax.set_title(f'Confusion Matrix — {best_name}', fontweight='bold', pad=14, fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

### 6.3 5-Fold Stratified Cross-Validation

Cross-validation gives a more robust accuracy estimate by testing on every fold of the data.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
print('5-Fold Stratified Cross-Validation (F1-Macro)')
print('=' * 55)
for name, r in results.items():
    scores = cross_val_score(r['pipeline'], X, y_cat, cv=cv, scoring='f1_macro', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:22s}  {scores.mean():.3f} +/- {scores.std():.3f}   folds: {np.round(scores, 3)}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(
    [cv_results[m] for m in model_names],
    labels=model_names, patch_artist=True, notch=False, widths=0.5
)
for patch, color in zip(bp['boxes'], bar_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
for median in bp['medians']:
    median.set_color('black')
    median.set_linewidth(2)
ax.set_title('5-Fold CV F1-Macro Distribution', fontweight='bold', pad=12)
ax.set_ylabel('F1-Score (Macro)')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cross_validation.png', bbox_inches='tight')
plt.show()
print('Saved: cross_validation.png')

### 6.4 Top TF-IDF Features per Category

The highest-weight features reveal which words most strongly signal each category.

In [ ]:
best_pipeline = results[best_name]['pipeline']
tfidf_step    = best_pipeline.named_steps['tfidf']
clf_step      = best_pipeline.named_steps['clf']
feature_names = np.array(tfidf_step.get_feature_names_out())

# Works for both LogisticRegression and LinearSVC (both expose .coef_)
if hasattr(clf_step, 'coef_'):
    classes    = clf_step.classes_ if hasattr(clf_step, 'classes_') else LABELS
    fig, axes2 = plt.subplots(3, 3, figsize=(16, 12))
    axes2_flat = axes2.flatten()

    for idx, (cat, coefs) in enumerate(zip(classes, clf_step.coef_)):
        ax     = axes2_flat[idx]
        top_i  = np.argsort(coefs)[-10:]
        ax.barh(
            list(feature_names[top_i]),
            list(coefs[top_i]),
            color=palette[idx], alpha=0.85, edgecolor='white'
        )
        ax.set_title(str(cat).upper().replace('_', ' '), fontweight='bold', fontsize=10)
        ax.set_xlabel('TF-IDF Coefficient')
        ax.tick_params(axis='y', labelsize=8)

    for ax in axes2_flat[len(classes):]:
        ax.set_visible(False)

    plt.suptitle(f'Top 10 Discriminative Features per Category — {best_name}',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'feature_importance.png', bbox_inches='tight')
    plt.show()
    print('Saved: feature_importance.png')
else:
    print('Feature importance not available for this model type.')

---
## 7. Classifier vs. LLM Agent — Comparison

The trained classifier and the full LLM-based triage agent are complementary tools.

In [ ]:
comparison_data = {
    'Dimension': [
        'Model type',
        'Task coverage',
        'Speed',
        'Cost',
        'Accuracy (category)',
        'Reply drafting',
        'Policy lookup',
        'Escalation detection',
        'Explainability',
        'Offline use',
    ],
    f'Trained Classifier ({best_name})': [
        'TF-IDF + ' + best_name,
        'Category + Priority prediction only',
        '< 1 ms per email',
        'Free (runs locally)',
        f'{results[best_name]["f1_macro"]:.1%} F1-macro',
        'No',
        'No',
        'No',
        'High (feature weights visible)',
        'Yes',
    ],
    'LLM Agent (Gemini)': [
        'Large Language Model (zero-shot)',
        'Full triage pipeline',
        '3-5 seconds per email (API call)',
        'API credits per call',
        '~90% (agent confidence scores)',
        'Yes — policy-grounded draft reply',
        'Yes — real-time policy.md lookup',
        'Yes — with ESC rules and safety net',
        'Medium (step log explains decisions)',
        'No (requires API key)',
    ],
}

df_cmp = pd.DataFrame(comparison_data).set_index('Dimension')
display(df_cmp)

print('\nKey Insight:')
print('  The trained classifier is fast, free, and interpretable — ideal as a first-pass')
print('  pre-filter to sort high volumes of emails.')
print('  The LLM agent handles the full task: reply drafting, policy lookup, escalation')
print('  reasoning — capabilities a simple classifier cannot replicate.')
print('  Together they form a practical two-tier triage system.')

In [ ]:
# Radar / spider chart: qualitative capability comparison
import matplotlib.patches as mpatches

dimensions  = ['Classification\nAccuracy', 'Speed', 'Reply\nDrafting', 'Policy\nLookup', 'Explainability', 'Offline\nUse']
classifier_scores = [0.85, 1.0,  0.0,  0.0,  0.9, 1.0]   # normalised 0-1
llm_scores        = [0.90, 0.3,  1.0,  1.0,  0.5, 0.0]

angles = np.linspace(0, 2 * np.pi, len(dimensions), endpoint=False).tolist()
angles += angles[:1]
classifier_scores += classifier_scores[:1]
llm_scores        += llm_scores[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.plot(angles, classifier_scores, 'o-', linewidth=2, color='#5C6BC0', label='Trained Classifier')
ax.fill(angles, classifier_scores, alpha=0.20, color='#5C6BC0')
ax.plot(angles, llm_scores, 's-', linewidth=2, color='#EF5350', label='LLM Agent')
ax.fill(angles, llm_scores, alpha=0.20, color='#EF5350')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(dimensions, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8)
ax.set_title('Classifier vs LLM Agent — Capability Profile',
             fontweight='bold', pad=20, fontsize=12)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'capability_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: capability_comparison.png')

---
## 8. Summary & Conclusions

### What we built
- Loaded and merged **two datasets** (20 real + 400 synthetic student helpdesk emails).
- Performed **EDA**: category distribution, priority breakdown, class imbalance analysis, text length distribution, and top word frequencies per category.
- **Preprocessed** text: lowercasing, punctuation removal, short-token filtering.
- Trained **three classifiers** (Logistic Regression, Naive Bayes, Linear SVM) using TF-IDF features.
- Evaluated with **accuracy, F1-macro, F1-weighted, classification report, confusion matrix, and 5-fold cross-validation**.
- Visualised **feature importance** to explain which words drive each prediction.

### Key findings
1. **Best model** was the one with the highest F1-macro score (see Section 5).
2. **Class imbalance** is moderate (IR < 10); addressed with `class_weight='balanced'`.
3. Categories like `security`, `harassment`, and `wellbeing` are highly distinguishable from their unique vocabulary.
4. `fee` and `academic` share overlapping vocabulary (portal, university, office), making them the hardest to separate.

### Limitations
- Synthetic data lacks real-world noise and diversity.
- The classifier only predicts category and priority; it cannot draft replies or look up policy — that remains the LLM agent's role.

### Plots saved to `analysis/plots/`
- `category_distribution.png`
- `priority_distribution.png`
- `class_imbalance.png`
- `text_length.png`
- `word_frequency.png`
- `model_comparison.png`
- `confusion_matrix.png`
- `cross_validation.png`
- `feature_importance.png`
- `capability_comparison.png`